# Milestone 1: Data Profiling

Profil awal dataset besar Indonesia menggunakan Polars LazyFrame dan DuckDB. Isi sumber serta deskripsi dataset pada `data/README.md` sebelum menjalankan notebook ini.

In [2]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'playstore_com.whatsapp_2026-05-20_12-10-41.csv'

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset tidak ditemukan: {DATA_PATH}. '
        'Unduh data ke data/raw/ lalu sesuaikan DATA_PATH.'
    )

print(f'Dataset: {DATA_PATH.name}')
print(f'Ukuran file: {DATA_PATH.stat().st_size / 1_000_000:.1f} MB')

Dataset: playstore_com.whatsapp_2026-05-20_12-10-41.csv
Ukuran file: 760.7 MB


## Profil Polars

`scan_parquet` hanya membangun rencana query; file baru dibaca ketika `collect()` dipanggil.

In [3]:
lazy_data = pl.scan_csv(DATA_PATH)
print(lazy_data.collect_schema())

ringkasan = lazy_data.select([
    pl.len().alias('jumlah_baris'),
    pl.all().null_count().name.suffix('_null'),
]).collect()
ringkasan

Schema({'content': String, 'score': Int64, 'userName': String, 'at': String, 'replyContent': String, 'thumbsUpCount': Int64, 'reviewId': String, 'app_id': String})


jumlah_baris,content_null,score_null,userName_null,at_null,replyContent_null,thumbsUpCount_null,reviewId_null,app_id_null
u32,u32,u32,u32,u32,u32,u32,u32,u32
6187000,95,0,0,0,6186998,0,0,0


In [4]:
contoh_data = lazy_data.head(10).collect()
contoh_data

content,score,userName,at,replyContent,thumbsUpCount,reviewId,app_id
str,i64,str,str,str,i64,str,str
"""bagus""",5,"""Yongki gl""","""2026-05-19 12:10:17""",null,0,"""987ac248-4a41-48f3-b842-ccc24e…","""com.whatsapp"""
"""kenapa apk WA saya tidak bisa …",1,"""Cantona Sitorus""","""2026-05-19 12:07:45""",null,0,"""8bc4f110-e424-4e63-a741-b42225…","""com.whatsapp"""
"""Saya suka""",5,"""Angelina Herlina mi""","""2026-05-19 12:07:40""",null,0,"""f2619c34-b09c-496d-add9-34cc35…","""com.whatsapp"""
"""bagus""",5,"""Diastamar Ma'ruf""","""2026-05-19 12:05:53""",null,0,"""6dd902d1-09cc-4f38-9be4-9b9b28…","""com.whatsapp"""
"""bagus, nyaman""",5,"""Nabila Aulia Husna""","""2026-05-19 12:04:40""",null,0,"""beeeabf4-be01-42a1-a3ba-996180…","""com.whatsapp"""
"""nomor telepon aku tidak bisa m…",1,"""ilham ART""","""2026-05-19 12:04:32""",null,0,"""9d2faa60-6e45-472f-b581-a9453e…","""com.whatsapp"""
"""apk WhatsApp sangat membabtu""",5,"""Kikikurnia Kurnia""","""2026-05-19 12:04:29""",null,0,"""95ac98a3-c045-4a4d-b219-c1044e…","""com.whatsapp"""
"""sangat puas dengan apk sekaran…",5,"""Yuda Bastrad""","""2026-05-19 12:04:23""",null,0,"""7bcc243d-1bda-499f-8934-439f06…","""com.whatsapp"""
"""banyak bug wak tag orang video…",4,"""gusrokk 887""","""2026-05-19 12:03:23""",null,0,"""ef89b724-63f6-48ac-9bb6-c25a1b…","""com.whatsapp"""


## Profil DuckDB

DuckDB membaca file Parquet langsung dari disk dan menghasilkan ringkasan statistik setiap kolom.

In [6]:
connection = duckdb.connect()
profil_duckdb = connection.execute(
    "SUMMARIZE SELECT * FROM read_csv_auto(?)", [str(DATA_PATH)]
).pl()
profil_duckdb

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""content""","""VARCHAR""","""!""","""🫸🫷🩷""",2745991,null,null,null,null,null,6187000,0.00
"""score""","""BIGINT""","""1""","""5""",5,"""4.084312429287215""","""1.4678998506993732""","""4""","""5""","""5""",6187000,0.00
"""userName""","""VARCHAR""","""!!!!!!???????""","""흫zulzz""",4395338,null,null,null,null,null,6187000,0.00
"""at""","""TIMESTAMP""","""2019-05-25 03:26:11""","""2026-05-19 12:10:17""",5582592,"""2022-01-15 03:01:00.160492""",null,"""2020-06-18 23:45:25.41954""","""2021-08-20 22:10:28.651697""","""2023-07-02 00:18:14.254576""",6187000,0.00
"""replyContent""","""VARCHAR""","""Satu-satunya cara untuk memuli…","""Versi terbaru memiliki sejumla…",2,null,null,null,null,null,6187000,100.00
"""thumbsUpCount""","""BIGINT""","""0""","""77842""",2061,"""1.4070056570227898""","""92.10772648722524""","""0""","""0""","""0""",6187000,0.00
"""reviewId""","""VARCHAR""","""000005b6-f39f-4212-935c-23fa15…","""fffffe48-6853-4227-9de2-4c294d…",6002195,null,null,null,null,null,6187000,0.00
"""app_id""","""VARCHAR""","""com.whatsapp""","""com.whatsapp""",1,null,null,null,null,null,6187000,0.00


## Catatan Milestone 1

Tulis temuan awal di bawah ini: ukuran data, jumlah baris, tipe kolom, kolom dengan null terbanyak, serta anomali yang perlu ditangani pada Milestone 2.

### Catatan Hasil Data Profiling (Milestone 1)

Berdasarkan hasil eksekusi Polars dan DuckDB, berikut adalah ringkasan temuan awal:

1. **Dimensi dan Tipe Data**: Dataset terdiri dari 6.187.000 baris dan 8 kolom. DuckDB secara otomatis mendeteksi tipe data teks/`VARCHAR` (`content`, `userName`, `replyContent`, `reviewId`, `app_id`), numerik/`BIGINT` (`score`, `thumbsUpCount`), dan mengenali kolom `at` sebagai `TIMESTAMP`. Rentang waktu ulasan terdeteksi cukup panjang, membentang dari 25 Mei 2019 hingga 19 Mei 2026.
2. **Identifikasi Null**: Kolom `replyContent` memiliki persentase *null* absolut sebesar 100.00%. 
3. **Anomali dan Rencana Tindakan untuk Milestone 2 (Data Cleaning)**:
   - **Duplikasi Data**: Jumlah baris total adalah 6.187.000, tetapi kolom `reviewId` hanya mencatat nilai unik (*approx_unique*) sekitar 6.002.195. Terdapat selisih ratusan ribu baris yang terindikasi sebagai data ganda (*duplicate rows*) dan wajib dibersihkan menggunakan metode *drop duplicates* berbasis `reviewId`.
   - **Penghapusan Kolom (Drop Columns)**: 
     - Kolom `replyContent` akan dihapus karena 100% kosong.
     - Kolom `app_id` juga akan dihapus karena kardinalitasnya hanya 1 (semua baris berisi "com.whatsapp").
4. **Potensi Analisis Lanjutan**: Mengingat besarnya volume teks ulasan (`content`), kumpulan data ini sangat ideal untuk difilter lebih spesifik nantinya menggunakan pencarian kata kunci Polars. Ekstraksi opini terkait privasi, keamanan, atau fitur pelacakan lokasi (*live location*) dapat dilakukan dengan mudah pada fase *insight* analisis.